# 12 — Ablation CNN 2D et cartes d'activation

Chaque expérience modifie un seul facteur par rapport à la référence : padding, pas de la première convolution, type de pooling, capacité ou convolution ponctuelle 1 × 1. Tous les autres hyperparamètres restent ceux de `src.config`. Le classement des architectures repose exclusivement sur le macro-F1 de validation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import *
from src.helpers import count_parameters, set_seed

set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Périphérique sélectionné :", device)
print("CUDA disponible :", torch.cuda.is_available())

In [ ]:
from src.models import FlattenedScalogramMLP, WESADScalogramCNN
from src.scalograms import WESADScalogramDataset
from src.experiments import run_validation_selected_experiment
from src.visualization import extract_feature_maps

scalogram_dir = PROJECT_ROOT / "data/processed/scalograms"
metadata_dir = PROJECT_ROOT / "data/processed/metadata"
ABlations = {
    "baseline": dict(padding=1, conv_stride=1, pooling_type="max", filters=(16, 32, 64), use_pointwise_conv=False),
    "padding_0": dict(padding=0, conv_stride=1, pooling_type="max", filters=(16, 32, 64), use_pointwise_conv=False),
    "stride_2_first_conv": dict(padding=1, conv_stride=2, pooling_type="max", filters=(16, 32, 64), use_pointwise_conv=False),
    "avg_pool": dict(padding=1, conv_stride=1, pooling_type="avg", filters=(16, 32, 64), use_pointwise_conv=False),
    "filters_8_16_32": dict(padding=1, conv_stride=1, pooling_type="max", filters=(8, 16, 32), use_pointwise_conv=False),
    "pointwise_1x1": dict(padding=1, conv_stride=1, pooling_type="max", filters=(16, 32, 64), use_pointwise_conv=True),
}
pd.DataFrame(ABlations).T

Le pas 2 est appliqué uniquement à la première convolution. La référence avec pas 1, les deux valeurs de padding, les deux poolings, les deux capacités et l'ajout 1 × 1 forment des comparaisons contrôlées.

In [ ]:
RUN_ABLATIONS = False
ablation_results = {}
if RUN_ABLATIONS:
    datasets = tuple(WESADScalogramDataset(
        scalogram_dir / f"X_{split}.pt",
        scalogram_dir / f"y_{split}.pt",
        metadata_dir / f"windows_{split}.csv",
    ) for split in ("train", "validation", "test"))
    validation_meta = pd.read_csv(metadata_dir / "windows_validation.csv")
    test_meta = pd.read_csv(metadata_dir / "windows_test.csv")
    for name, architecture in ABlations.items():
        factory = lambda architecture=architecture: WESADScalogramCNN(**architecture)
        ablation_results[name] = run_validation_selected_experiment(
            factory, datasets, validation_meta, test_meta,
            PROJECT_ROOT / "artifacts/models/cnn2d_ablations" / name,
            {
                "model_class": "WESADScalogramCNN", "architecture": architecture,
                "seed": RANDOM_SEED, "subject_split": SPLIT_SUBJECTS,
                "input_channels": SCALOGRAM_CHANNELS, "input_shape": [3, 64, 64],
                "normalization_statistics": "artifacts/preprocessing/scalograms/scalogram_metadata.json",
            }, device, compare_weighted_loss=False,
        )
else:
    print("Non exécuté : activer RUN_ABLATIONS après le notebook 10.")

## Tableau d'ablation et comparaison avec le MLP sur scalogrammes

In [ ]:
def artifact_row(name, path):
    with open(path / "model_config.json", encoding="utf-8") as f: config = json.load(f)
    with open(path / "validation_metrics.json", encoding="utf-8") as f: val = json.load(f)
    with open(path / "test_metrics.json", encoding="utf-8") as f: test = json.load(f)
    with open(path / "training_summary.json", encoding="utf-8") as f: summary = json.load(f)
    arch = config.get("architecture", {})
    return {
        "model": name, "input representation": config.get("input_shape", "raw signals"),
        "padding": arch.get("padding"), "stride": arch.get("conv_stride"),
        "pooling": arch.get("pooling_type"), "filters": arch.get("filters"),
        "1x1 convolution": arch.get("use_pointwise_conv"),
        "parameter count": config.get("parameter_count"), "best epoch": summary.get("best_epoch"),
        "validation macro F1": val.get("macro_f1"), "test macro F1": test.get("macro_f1"),
        "stress precision": test.get("stress_precision"), "stress recall": test.get("stress_recall"),
        "ROC-AUC": test.get("roc_auc"), "average precision": test.get("average_precision"),
        "training time": summary.get("training_time_seconds"),
        "test inference time": test.get("inference_time_seconds"),
    }

paths = {name: PROJECT_ROOT / "artifacts/models/cnn2d_ablations" / name for name in ABlations}
paths.update({"flattened_scalogram_mlp": PROJECT_ROOT / "artifacts/models/scalogram_mlp"})
rows = [artifact_row(name, path) for name, path in paths.items() if (path / "model_config.json").exists()]
ablation_table = pd.DataFrame(rows)
display(ablation_table.sort_values("validation macro F1", ascending=False) if len(ablation_table) else "Aucun artefact d'ablation.")

## Cartes d'activation

Les hooks sont retirés immédiatement après l'inférence. Les images ci-dessous sont des activations apprises, et non une preuve physiologique directe. Une formulation prudente est : « cette carte répond fortement à un motif temps-fréquence localisé ». Les réponses peuvent coïncider avec des structures BVP périodiques, des variations EDA lentes, une activité d'accélération rapide ou des bandes larges/étroites, sans autoriser une interprétation causale.

In [ ]:
RUN_FEATURE_MAPS = False
if RUN_FEATURE_MAPS:
    dataset = WESADScalogramDataset(scalogram_dir / "X_validation.pt", scalogram_dir / "y_validation.pt", metadata_dir / "windows_validation.csv")
    example, label, metadata = dataset[0]
    model = WESADScalogramCNN().to(device)
    model.load_state_dict(torch.load(PROJECT_ROOT / "artifacts/models/cnn2d/best_model.pt", map_location=device, weights_only=True))
    maps = extract_feature_maps(model, example[None], [model.features[0], model.features[8]])
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for i, channel in enumerate(SCALOGRAM_CHANNELS):
        axes[i].imshow(example[i], aspect="auto", origin="lower", cmap="viridis"); axes[i].set_title(channel)
    plt.show()
    for title, activation in zip(["Première convolution", "Dernier bloc convolutif"], maps):
        count = min(8, activation.shape[1]); fig, axes = plt.subplots(2, 4, figsize=(12, 6));
        for i, ax in enumerate(axes.flat):
            ax.axis("off")
            if i < count: ax.imshow(activation[0, i], aspect="auto", origin="lower", cmap="magma"); ax.set_title(f"Carte {i+1}")
        fig.suptitle(title); plt.show()